# 🛠️ Notebook 2: ATM — Implementation

## 🛠️ Setup

```bash
cd 07-object-oriented-design/atm
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


In [ ]:
from dataclasses import dataclass
from enum import Enum

class State(Enum):
    IDLE = 0; CARD_INSERTED = 1; AUTHENTICATED = 2

@dataclass
class Account:
    id: str
    pin: str
    balance: float

@dataclass
class Card:
    number: str
    account: Account

class CashDispenser:
    def __init__(self, cash: float):
        self.cash = cash
    def dispense(self, amt):
        if amt > self.cash: raise RuntimeError('ATM empty')
        self.cash -= amt
    def accept(self, amt):
        self.cash += amt

class ATM:
    def __init__(self, dispenser: CashDispenser):
        self.state = State.IDLE
        self.dispenser = dispenser
        self.card = None
        self.log = []

    def insert_card(self, card: Card):
        if self.state != State.IDLE: raise RuntimeError('busy')
        self.card = card; self.state = State.CARD_INSERTED

    def enter_pin(self, pin: str):
        if self.state != State.CARD_INSERTED: raise RuntimeError('insert card first')
        if self.card.account.pin != pin:
            self.eject(); raise RuntimeError('wrong PIN — card ejected')
        self.state = State.AUTHENTICATED

    def _require_auth(self):
        if self.state != State.AUTHENTICATED: raise RuntimeError('not authenticated')

    def balance(self):
        self._require_auth(); return self.card.account.balance

    def withdraw(self, amt):
        self._require_auth()
        acc = self.card.account
        if amt > acc.balance: raise RuntimeError('insufficient funds')
        self.dispenser.dispense(amt)
        acc.balance -= amt
        self.log.append(('withdraw', amt))

    def deposit(self, amt):
        self._require_auth()
        self.dispenser.accept(amt)
        self.card.account.balance += amt
        self.log.append(('deposit', amt))

    def eject(self):
        self.state = State.IDLE; self.card = None


## Use the ATM

In [ ]:
acc = Account('A-1', '1234', 500)
card = Card('CARD-1', acc)
atm = ATM(CashDispenser(cash=1000))

atm.insert_card(card)
atm.enter_pin('1234')
print('balance:', atm.balance())
atm.withdraw(200)
atm.deposit(50)
print('balance after:', atm.balance())
print('ATM cash:', atm.dispenser.cash)
print('log:', atm.log)
atm.eject()


### Error paths

In [ ]:
# Wrong PIN ejects the card
atm.insert_card(card)
try: atm.enter_pin('9999')
except RuntimeError as e: print('expected:', e)
print('state after bad pin:', atm.state)

# Withdraw without auth
atm.insert_card(card)
try: atm.withdraw(10)
except RuntimeError as e: print('expected:', e)
atm.eject()

# Overdraw
atm.insert_card(card); atm.enter_pin('1234')
try: atm.withdraw(99999)
except RuntimeError as e: print('expected:', e)


### Extensions
- Rate-limit failed PIN attempts (block card after 3).
- Support multiple accounts per card (checking/savings).
- Concurrency — multiple ATMs on one account.